In [2]:
# ============================================================================
# BLOCK 5: TOPIC MODELING - POLISH
# Input: 04_sentiment_data_pl.pkl
# Output: 05_topics_data_pl.pkl
# Runtime: ~15-20 minutes on CPU
# ============================================================================
%run 00_setup_and_config.ipynb

c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\election_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 11:35:24
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_ana

In [3]:
# CELL 1: Load previous checkpoint
print('='*70)
print('BLOCK 5: TOPIC MODELING - POLISH')
print('='*70)

if check_checkpoint_exists('pl', '05_topics_data'):
    df_pl = load_checkpoint('pl', '05_topics_data')
    topic_model = None
    print('✓ Loading from topics checkpoint (skipping modeling)')
else:
    df_pl = load_checkpoint('pl', '04_sentiment_data')
    if df_pl is None:
        raise FileNotFoundError('Run 03_sentiment_analysis_pl.ipynb first')

print(f'\nRunning BERTopic on {len(df_pl):,} Polish comments...')
print(f'Target topics: {BERTOPIC_CONFIG["n_topics"]}')
print(f'Embedding model: {EMBEDDING_MODEL}')

BLOCK 5: TOPIC MODELING - POLISH
✓ Loading checkpoint: 04_sentiment_data_pl.pkl

Running BERTopic on 6,763 Polish comments...
Target topics: 8
Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [4]:
# CELL 2: Load embedding model
print('\nLoading embedding model...')
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embedding_model = embedding_model.to('cpu')
print('✓ Embedding model loaded')


Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2353.08it/s]


✓ Embedding model loaded


In [5]:
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
import gc

gc.collect()

# ============================================================================
# CREATE VECTORIZER WITH POLISH STOPWORDS
# ============================================================================
print('\nCreating CountVectorizer with Polish stopwords...')

vectorizer_pl = CountVectorizer(
    stop_words=STOPWORDS_PL,          # ← From 00_setup_and_config.ipynb
    ngram_range=(2, 4),               # Only phrases (no single words)
    min_df=2,                         # Must be <= number of topics after reduction
    max_df=0.95,                      # Allow words in up to 95% of documents
    lowercase=True,
    token_pattern=r'(?u)\b\w[\w-]*\w\b|\w+\b'
)

print(f'✓ CountVectorizer created with {len(STOPWORDS_PL)} stopwords')

# ============================================================================
# CREATE BERTOPIC MODEL
# ============================================================================
print('\nInitializing BERTopic...')

topic_model = BERTopic(
    language='multilingual',
    embedding_model=embedding_model,
    nr_topics=8,                      # Target ~8 topics
    min_topic_size=15,                # Minimum documents per topic
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_pl,   # ← Custom vectorizer with stopwords
    representation_model=KeyBERTInspired()
)

print('✓ BERTopic model initialized')

# ============================================================================
# PREPARE TEXTS
# ============================================================================
print('\nPreparing texts...')

texts = df_pl['clean_text'].fillna('').astype(str).tolist()

# Filter very short texts (less than 3 words)
texts_filtered = []
indices_kept = []
for i, t in enumerate(texts):
    if len(t.split()) >= 3:
        texts_filtered.append(t)
        indices_kept.append(i)

print(f'  Original texts: {len(texts):,}')
print(f'  After filtering: {len(texts_filtered):,}')
print(f'  Removed: {len(texts) - len(texts_filtered):,} short texts')

# ============================================================================
# FIT MODEL
# ============================================================================
print('\nFitting BERTopic model...')
topics, probs = topic_model.fit_transform(texts_filtered)

# Map back to original dataframe
df_pl['topic'] = -1
df_pl['topic_probability'] = 0.0

for idx, (topic, prob) in zip(indices_kept, zip(topics, probs)):
    df_pl.loc[idx, 'topic'] = topic
    df_pl.loc[idx, 'topic_probability'] = max(prob)

# Save topic model
topic_model.save(PROCESSED_DIR / 'bertopic_model_pl')
print(f'✓ Topic model saved')

# Save checkpoint
save_checkpoint(df_pl, 'pl', '05_topics_data')
update_pipeline_status('pl', 5, 'completed')

# ============================================================================
# QUALITY CHECK
# ============================================================================
print('\n' + '='*70)
print('TOPIC QUALITY REPORT')
print('='*70)

topic_info = topic_model.get_topic_info()
display(topic_info)

# Count outliers
outliers = (df_pl['topic'] == -1).sum()
outlier_pct = 100 * outliers / len(df_pl)
print(f'\n⚠️  OUTLIERS (Topic -1): {outliers:,} comments ({outlier_pct:.1f}%)')

if outlier_pct > 30:
    print('   ⚠️  WARNING: High outlier rate - consider reducing min_topic_size')
else:
    print('   ✓ Outlier rate acceptable')

# ============================================================================
# SHOW TOPIC WORDS (FIXED ITERATION)
# ============================================================================
print('\n--- TOP 5 WORDS PER TOPIC ---')

for _, row in topic_info.iterrows():
    topic_id = int(row['Topic'])
    
    # Skip outliers
    if topic_id == -1:
        continue
    
    # Get topic words
    topic_words = topic_model.get_topic(topic_id)
    
    # Safety check
    if not topic_words or not isinstance(topic_words, list):
        print(f"⚠️  Topic {topic_id}: Could not retrieve words")
        continue
    
    words = [w for w, s in topic_words[:5]]
    count = int(row['Count'])
    print(f"Topic {topic_id} ({count:,} comments): {', '.join(words)}")

2026-05-29 11:35:31,911 - BERTopic - Embedding - Transforming documents to embeddings.



Creating CountVectorizer with Polish stopwords...
✓ CountVectorizer created with 554 stopwords

Initializing BERTopic...
✓ BERTopic model initialized

Preparing texts...
  Original texts: 6,763
  After filtering: 6,445
  Removed: 318 short texts

Fitting BERTopic model...


Batches: 100%|██████████| 202/202 [02:22<00:00,  1.42it/s]
2026-05-29 11:37:53,995 - BERTopic - Embedding - Completed ✓
2026-05-29 11:37:53,995 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-29 11:38:24,838 - BERTopic - Dimensionality - Completed ✓
2026-05-29 11:38:24,838 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-29 11:38:26,018 - BERTopic - Cluster - Completed ✓
2026-05-29 11:38:26,018 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-29 11:38:26,669 - BERTopic - Representation - Completed ✓
2026-05-29 11:38:26,669 - BERTopic - Topic reduction - Reducing number of topics
2026-05-29 11:38:26,684 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-29 11:38:31,405 - BERTopic - Representation - Completed ✓
2026-05-29 11:38:31,405 - BERTopic - Topic reduction - Reduced number of topics from 53 to 8
2026-05-29 11:38:33,600 - BERTopic - WARNING: When 

✓ Topic model saved
✓ Checkpoint saved: 05_topics_data_pl.pkl
✓ Pipeline status updated: pl - Block 5 - completed

TOPIC QUALITY REPORT


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2167,-1_pana stanowskiego_pana nawrockiego_pana redaktora_panie redaktorze,"[pana stanowskiego, pana nawrockiego, pana redaktora, panie redaktorze, pan stanowski, pan nawro...","[Panie Krzysztofie, proszę rozważyć kandydowanie w następnej kadencji. Tym razem na poważnie. Br..."
1,0,2948,0_brawo panie prezydencie_panie prezydencie brawo_brawo panie prezydencie brawo_brawo dla prezyd...,"[brawo panie prezydencie, panie prezydencie brawo, brawo panie prezydencie brawo, brawo dla prez...","[Brawo Panie Prezydencie, brawo Panie Prezydencie!, Brawo Panie Prezydencie!]"
2,1,576,1_dziekuje panie_pozdrawiam panie_pozdrawiam serdecznie pana_pozdrawiam pana,"[dziekuje panie, pozdrawiam panie, pozdrawiam serdecznie pana, pozdrawiam pana, pozdrawiam pana ...","[Panie Rafale, dziękuję z bardzo trafnną analizę. Jednak mało optymistyczną na przyszłość, Dzięk..."
3,2,311,2_mld złotych_wprowadzić euro_185 mld_tego kredytu,"[mld złotych, wprowadzić euro, 185 mld, tego kredytu, chodzi pieniądze, 300 miliardów, wpływy bu...","[NBP nie osiągnęło 185 mld zł zysku z posiadanych rezerw złota,ta kwota to nie zysk a zwrostu wa..."
4,3,224,3_ukraińców prostu_dla ukraińców_dla ukrainy_od niemiec,"[ukraińców prostu, dla ukraińców, dla ukrainy, od niemiec, wojna ukrainie, ukraincy ukraina, oby...","[Brawo panie prezydencie RP.... Niemcy niech okradają innych, Panie Redaktorze tutaj wciąż zacho..."
5,4,151,4_założy dom medialny_domów medialnych_domu medialnego_dom medialny,"[założy dom medialny, domów medialnych, domu medialnego, dom medialny, stanowski założy dom medi...","[Niech prawicowe media otworzą założą własny dom medialny., Dziękujemy Panie Macieju Podstawka z..."
6,5,37,5_tą ustawę_tej ustawie_tym temacie_tym co,"[tą ustawę, tej ustawie, tym temacie, tym co, inna sprawa, tego ludzie, co tego, tylko mówiąc, w...",[Ważne: 1.1. Zwiększając akcyzę doprowadza się do większych wydatków dla konsumenta lub produkcj...
7,6,31,6_koń uśmiał_koń trojański_historia zatoczyła_słychać wycie,"[koń uśmiał, koń trojański, historia zatoczyła, słychać wycie, tkwi tym, giertych bedzie, tym pr...","[Giertych bedzie siedzial to kwestia czasu, koniu tak bedzie zrobiony ze zostanie PiSu sympatyki..."



⚠️  OUTLIERS (Topic -1): 2,603 comments (37.8%)
   ⚠️  WARNING: High outlier rate - consider reducing min_topic_size

--- TOP 5 WORDS PER TOPIC ---
Topic 0 (2,948 comments): brawo panie prezydencie, panie prezydencie brawo, brawo panie prezydencie brawo, brawo dla prezydenta, prezydencie brawo
Topic 1 (576 comments): dziekuje panie, pozdrawiam panie, pozdrawiam serdecznie pana, pozdrawiam pana, pozdrawiam pana serdecznie
Topic 2 (311 comments): mld złotych, wprowadzić euro, 185 mld, tego kredytu, chodzi pieniądze
Topic 3 (224 comments): ukraińców prostu, dla ukraińców, dla ukrainy, od niemiec, wojna ukrainie
Topic 4 (151 comments): założy dom medialny, domów medialnych, domu medialnego, dom medialny, stanowski założy dom medialny
Topic 5 (37 comments): tą ustawę, tej ustawie, tym temacie, tym co, inna sprawa
Topic 6 (31 comments): koń uśmiał, koń trojański, historia zatoczyła, słychać wycie, tkwi tym


In [6]:
# CELL 4: Topic summary
print('\n' + '='*70)
print('TOPIC SUMMARY - POLISH')
print('='*70)

topic_info.to_csv(OUTPUT_DIR / 'polish' / 'pl_topics_summary.csv', index=False)
print(f"\n✓ Saved: {OUTPUT_DIR / 'polish' / 'pl_topics_summary.csv'}")


TOPIC SUMMARY - POLISH

✓ Saved: outputs\polish\pl_topics_summary.csv


In [8]:
# CELL 5: Topic visualization
print('\nGenerating topic visualization...')

fig, ax = plt.subplots(figsize=(14, 8))

# Get topic sizes
topic_sizes = topic_model.get_topic_info().sort_values('Count', ascending=False)

colors = plt.cm.Set3(np.linspace(0, 1, len(topic_sizes)))
bars = ax.barh(range(len(topic_sizes)), topic_sizes['Count'].values, color=colors)

ax.set_yticks(range(len(topic_sizes)))
ax.set_yticklabels([f"Topic {i}: {rep[:50]}..." 
                    for i, rep in zip(topic_sizes['Topic'], topic_sizes['Representation'])])
ax.set_xlabel('Number of Comments', fontsize=12)
ax.set_title(f'Topic Distribution - Polish (n={len(df_pl):,})', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
save_figure(fig, 'pl_topics_overview.png', 'pl', 'visualizations')



Generating topic visualization...
✓ Saved: outputs\pl\visualizations\pl_topics_overview.png


In [9]:
print('\n' + '='*70)
print('✓ BLOCK 5 COMPLETE - TOPIC MODELING DONE')
print('='*70)


✓ BLOCK 5 COMPLETE - TOPIC MODELING DONE
